# Sanket: INCLUDE-50 GRU baseline on Kaggle (Phase 2)

Thin notebook: every step calls code in `ml/src/sanket/`. Nothing here should contain logic worth testing.

**Before running**
1. The repository is public on GitHub, so no token is needed.
2. Add the private keypoints dataset (made by `make upload ONLY=include50`) as an input.
3. Settings: Accelerator = **GPU T4** (not P100; cell 5 stops if it isn't a T4), Internet = **on**.
4. Set `GIT_SHA` below to the commit to reproduce (every result must trace to a config + a SHA).

**Resuming after the 12 h limit:** attach this notebook's previous output as an input and set `RESUME_FROM`.
`sanket.train` continues from `last.pt` and reproduces the uninterrupted run exactly (tested).

In [ ]:
# 1. Parameters
REPO_URL = "https://github.com/octotat-bot/sanket.git"
GIT_SHA = "phase-1"  # a commit sha (or branch); the resolved sha is printed and saved in git_sha.txt
CONFIG = "configs/gru_include50.yaml"
RUN_NAME = "gru_include50"
RESUME_FROM = None  # e.g. "/kaggle/input/<previous-output>/runs/gru_include50"
WORK = "/kaggle/working"

In [ ]:
# 2. Code at the pinned commit
import os, subprocess
token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    pass  # public repo, or no secret configured
url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL
subprocess.run(["git", "clone", "--quiet", url, f"{WORK}/sanket"], check=True)
subprocess.run(["git", "-C", f"{WORK}/sanket", "checkout", "--quiet", GIT_SHA], check=True)
print(subprocess.run(["git", "-C", f"{WORK}/sanket", "rev-parse", "HEAD"], capture_output=True, text=True).stdout)

In [ ]:
# 3. Pinned environment (Python 3.12 + the locked `train` group: torch 2.15.0, tensorboard)
!pip install --quiet uv==0.11.13
%cd /kaggle/working/sanket/ml
!uv sync --locked --group train

In [ ]:
# 4. Keypoints: link the private dataset where the config expects it (data/keypoints/v1)
import glob, os
hits = sorted(glob.glob("/kaggle/input/**/_done_zips.txt", recursive=True))
assert hits, "keypoints dataset not found under /kaggle/input (add it as an input)"
src = os.path.dirname(hits[0])
dst = "/kaggle/working/sanket/data/keypoints/v1"
os.makedirs(os.path.dirname(dst), exist_ok=True)
if not os.path.exists(dst):
    os.symlink(src, dst)
print(src, "->", dst, "|", len(glob.glob(dst + "/**/*.npz", recursive=True)), "npz files")

In [ ]:
# 5. Sanity checks: a T4 GPU, versions, and the INCLUDE-50 split loads with the expected counts
# (subprocess, not "!": IPython would substitute the {...} in these one-liners)
import subprocess
GPU = ("import torch; n = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'; "
       "print('torch', torch.__version__, '| gpu', n); "
       "assert 'T4' in n, 'Settings > Accelerator: choose GPU T4 (not P100)'")
SPLITS = ("from pathlib import Path; from sanket.data.keypoint_index import load_split_index as L; "
          "d = Path('../data/keypoints/v1').resolve(); "
          "r = {s: (len(L('include50', s, d).items), len(L('include50', s, d).missing)) for s in ('train', 'val', 'test')}; "
          "print(r); assert r == {'train': (689, 0), 'val': (77, 0), 'test': (192, 0)}, r")
for code in (GPU, SPLITS):
    subprocess.run(["uv", "run", "--no-sync", "python", "-c", code], check=True)


In [ ]:
# 6. Train (resumable). Early stopping uses the val split carved from train, never test (ADR 003).
import os, shutil
run_dir = f"{WORK}/runs/{RUN_NAME}"
if RESUME_FROM and not os.path.exists(run_dir):
    shutil.copytree(RESUME_FROM, run_dir)
!uv run --no-sync python -m sanket.train --config {CONFIG} --run-dir {run_dir}

In [ ]:
# 7. Evaluate the best-validation checkpoint on val and on the official test split
!uv run --no-sync python -m sanket.evaluate --config {CONFIG} --run-dir {run_dir} --split val
!uv run --no-sync python -m sanket.evaluate --config {CONFIG} --run-dir {run_dir} --split test

Outputs are in `/kaggle/working/runs/<RUN_NAME>/`: `config.yaml`, `git_sha.txt`, `history.json`, `best.pt`,
`last.pt`, TensorBoard events (`tb/`) and `eval_val.json` / `eval_test.json`. Download them; the README
numbers are generated from these files by script, never copied by hand.